# v16 案B: PF全置換 — 微分可能DP / soft-Viterbi

PFは状態空間モデルの事後をサンプリング近似していた。離散グリッド上の厳密な対応物が
**forward-backward (sum-product)**。それを層にして end-to-end 学習する = PFの最も忠実なNN化。

- **emission(尤度)**: 水平GRの学習埋め込み と typewell埋め込みを `baseline±offset` のTVTでサンプルして内積。
  → 学習されたマルチスケールGR照合(窓)。
- **transition(dip動力学)**: 残差 offset の変化を罰する平滑性 `logT[k,k']=-(off_k-off_k')^2/(2 sig^2)`。
- **soft forward-backward**: 各位置の周辺事後 gamma を出し、期待 offset を読み出す(全て微分可能)。

状態 = baseline TVT (= s_base - Z) からの残差 offset。バンド幅 ±BAND ft、分解能 dt。
`TVT = (s_base - Z) + 期待offset`。 run_cv 互換のため `r_raw = 期待offset / TARGET_SCALE` を返す。

> 実行はユーザー。計算量は H(クロップ長)×K(状態数) の逐次DP。crop/K は VRAM と速度で調整。

In [1]:
import importlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import nn_pf_common as C
importlib.reload(C)
print('device:', C.DEVICE)

device: cuda


In [2]:
wells = C.build_all('train')
r_std, r_mad = C.target_r_std(wells)
print(f'residual r: std={r_std:.2f} ft, mad={r_mad:.2f} ft  -> BAND は r_std を十分覆う値に')

  built 100/773
  built 200/773
  built 300/773
  built 400/773
  built 500/773
  built 600/773
  built 700/773
[build_all] 773 wells ready (split=train)
residual r: std=42.40 ft, mad=26.39 ft  -> BAND は r_std を十分覆う値に


In [3]:
class ModelB(nn.Module):
    """微分可能DP。emission=学習GR照合, transition=残差平滑性。"""
    def __init__(self, gr_ch=32, band=60.0, dt=1.5):
        super().__init__()
        self.h_gr = C.GRConv(gr_ch)
        self.t_gr = C.GRConv(gr_ch)
        self.gr_ch = gr_ch
        K = int(round(2 * band / dt)) + 1
        off = (torch.arange(K, dtype=torch.float32) - K // 2) * dt   # [K] offset(ft)
        self.register_buffer('off', off)
        self.K = K
        self.log_temp  = nn.Parameter(torch.tensor(0.0))   # emission 温度 (softplus)
        self.log_sig   = nn.Parameter(torch.tensor(np.log(3.0)))  # transition 平滑性
        self.log_sig0  = nn.Parameter(torch.tensor(np.log(band / 3)))  # 初期prior
        self.emit_bias = nn.Parameter(torch.zeros(1))

    def emission(self, b):
        # 学習埋め込み
        hf = self.h_gr(b['gr_n']).transpose(1, 2)            # [B,C,H]
        tf = self.t_gr(b['tw_gr_n']).transpose(1, 2)         # [B,C,T]
        Bn, Cc, H = hf.shape
        base_tvt = (b['sbase'] - b['z'])                     # [B,H] baseline TVT
        query = base_tvt.unsqueeze(-1) + self.off.view(1, 1, -1)   # [B,H,K]
        q = query.reshape(Bn, H * self.K)                    # [B,H*K]
        tw_samp = C.batched_interp(b['tw_tvt'], tf, q)       # [B,C,H*K]
        tw_samp = tw_samp.reshape(Bn, Cc, H, self.K)
        e = (hf.unsqueeze(-1) * tw_samp).sum(1) / (Cc ** 0.5)  # [B,H,K]
        e = e * F.softplus(self.log_temp) + self.emit_bias
        return e                                            # [B,H,K]

    def forward(self, b):
        e = self.emission(b)                                 # [B,H,K]
        Bn, H, K = e.shape
        off = self.off
        sig = F.softplus(self.log_sig) + 1e-2
        sig0 = F.softplus(self.log_sig0) + 1e-2
        lT = -(off.view(K, 1) - off.view(1, K)) ** 2 / (2 * sig * sig)   # [K,K] (k',k)対称
        prior = -off ** 2 / (2 * sig0 * sig0)                            # [K]
        # --- forward (sum-product, log空間) ---
        alphas = []
        la = e[:, 0, :] + prior.view(1, K)
        alphas.append(la)
        for h in range(1, H):
            m = la.unsqueeze(2) + lT.unsqueeze(0)            # [B,K(k'),K(k)]
            la = e[:, h, :] + torch.logsumexp(m, dim=1)      # over k' -> [B,K]
            alphas.append(la)
        alpha = torch.stack(alphas, dim=1)                   # [B,H,K]
        # --- backward ---
        betas = [None] * H
        lb = torch.zeros(Bn, K, device=e.device)
        betas[H - 1] = lb
        for h in range(H - 2, -1, -1):
            m = lT.unsqueeze(0) + (e[:, h + 1, :] + lb).unsqueeze(1)  # [B,K(k),K(k')]
            lb = torch.logsumexp(m, dim=2)                   # over k' -> [B,K]
            betas[h] = lb
        beta = torch.stack(betas, dim=1)                     # [B,H,K]
        # --- 周辺事後 → 期待 offset ---
        gamma = torch.softmax(alpha + beta, dim=2)           # [B,H,K]
        exp_off = (gamma * off.view(1, 1, K)).sum(2)         # [B,H] (ft)
        return exp_off / C.TARGET_SCALE                      # run_cv 互換

def make_B():
    return ModelB(band=60.0, dt=1.5)

# 形状スモークテスト
from torch.utils.data import DataLoader
ds = C.WellDataset(wells[:8], mode='train', crop=384)
dl = DataLoader(ds, batch_size=4, collate_fn=C.collate)
b = C.to_device(next(iter(dl)))
m = make_B().to(C.DEVICE)
r = m(b)
print('K states:', m.K, '| r shape:', tuple(r.shape), '| pred:', tuple(C.tvt_from_r(r, b).shape))

K states: 81 | r shape: (4, 384) | pred: (4, 384)


In [ ]:
# CV 実行。DPは逐次なので crop は短め(384〜512)、batch も控えめから。
cv, oof = C.run_cv(wells, make_B, name='B_softdp',
                   n_splits=5, epochs=20, lr=1e-3, batch_size=4, crop=448)
print('CV overall RMSE (B):', cv)

c:\Users\kosaka256\Documents\rogii_claude\v16\nn_pf_common.py:335: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  tot += float(loss); nb += 1


[B_softdp] fold0 ep00 train_loss=196.1720 val_overall_rmse=38.0040
[B_softdp] fold0 ep01 train_loss=182.3218 val_overall_rmse=35.3257
[B_softdp] fold0 ep02 train_loss=160.4431 val_overall_rmse=36.9402
[B_softdp] fold0 ep03 train_loss=184.9715 val_overall_rmse=35.6143


In [ ]:
import os, pickle
os.makedirs('artifacts_v16', exist_ok=True)
with open('artifacts_v16/oof_B.pkl', 'wb') as f:
    pickle.dump({'cv': cv, 'oof': oof}, f)
print('saved artifacts_v16/oof_B.pkl')